In [ ]:
import numpy as np
import pandas as pd
import os


__mapping for categorical data conversion from orig mental health dataset__
To convert all categorical data to numerical, a dictionary was created for every cat feature with our chosen encodings.  
The convertData method takes a list of columns to drop from the dataframe, a list of the categorical feature mappings, and a location 
to export the converted dataframe to if necessary.  
The array of map dictionaries has to be in the order they appear in the original dataset, and there has to be one map for EVERY string variable
in the original dataset  

Imputation will be done in Orange to fill in missing values after the dataset is fully numerical. 

In [3]:
mapGender = {'Male':0, 'Female':1, 'Other':2}
mapGenre = {"Mobile Games":0, "MOBA":1, "FPS":2, "Battle Royale":3, "MMO":4, "RPG":5, "Strategy":6}
mapPrimary = {"Age of Empires":0, "Apex Legends":1, "CS:GO":2, "Call of Duty":3, "Candy Crush":4, "Civilization VI":5, 
              "Clash of Clans":6, "Cyberpunk 2077":7, "Dota 2":8, "Elden Ring":9, "Elder Scrolls Online":10, 
              "Final Fantasy XIV":11, "Fortnite":12, "Genshin Impact":13, "League of Legends":14, "Mobile Legends":15, 
              "Overwatch":16, "PUBG":17, "PUBG Mobile":18, "Skyrim":19, "StarCraft II":20, "Valorant":21, 
              "Warzone":22, "World of Warcraft":23 }
mapPlatform = {"PC":0, "Multi-platform":1, "Console":2, "Mobile":3}
mapSQual = {"Insomnia":0, "Very Poor":1, "Poor":2, "Fair":3, "Good":4}
mapSDis = {"Never":0, "Rarely":1, "Sometimes":2, "Often":3, "Always":4}
mapAperf = {"Failing":0, "Poor":1, "Below Average":2, "Average":3, "Good":4, "Excellent":5}
mapMood = {"Depressed":0, "Anxious":1, "Angry":2, "Irritable":3, "Withdrawn":4, "Restless":5, "Normal":6, "Excited":7, "Euphoric":8}
mapMSFreq = {"Never":0, "Rarely":1, "Sometimes":2, "Often":3, "Daily":4}
mapAddict = {"Low":0, "Moderate":1, "High":2, "Severe":3}
mapBinary = {"TRUE":1, "FALSE":0}
maps = [mapGender, mapGenre, mapPrimary, mapPlatform, mapSQual, mapSDis, mapAperf, mapMood, mapMSFreq, mapAddict]

In [4]:
def readData(datafile):
    # get the current directory: directory name for the abs path of the curr file
    dirpath = os.getcwd()
    abspath = dirpath + "\\..\\data\\" + datafile

    # read data into a pandas dataframe. data file is comma separated so use read_csv
    df = pd.read_csv(abspath)
     
    return df


# convert categorical data to numerical 
def convertData(data, drop:list[str], maps:list[dict], export:tuple[bool, str]):
    # drop indicated columns (should be a list of header names)
    if drop:
        for i in range(len(drop)):
            data = data.drop(columns=drop[i])

    # if the map array was provided, go through every feature in the dataframe and remap it using the 
    # corresponding dict in the map array if that feature is of string type
    if maps:
        # keep track of the number of converted columns to make sure the map is the correct one
        idx=0
        for feat in data.columns:
            # if the feature type is string, then its numerical and needs to be converted
            if data[feat].dtype == 'string':
                # map the feature to numerical and increase the index
                data[feat] = data[feat].map(maps[idx])
                idx += 1
            # pandas recognizes "true/True/TRUE" as booleans, but we need them as integers to convert to numpy
            # for every bool type feature, just infer the type to int to convert them to numerical
            elif data[feat].dtype == 'bool':
                data[feat] = data[feat].astype(int)

        # if the export tuple was provided and the first value is true, export the csv file to the specified
        # file location in the second tuple value
        if (export[0]):
            data.to_csv(export[1])

    # return the converted dataframe
    return data

In [5]:
filename = "GamingandMentalHealth.csv"  
data = readData(filename)

# drop the record id since it is unique to every sample
# convert all other categorical values to numeric using the defined mappings and export to a new csv file
dropCols = ["record_id"]
exportTo = (True, "../data/GamingandMentalHealth_conv_noid.csv")
data_conv = convertData(data.copy(), dropCols, maps, exportTo)

In [13]:
# running spearman correlation on selected subset

filename = "GamingAndMentalHealth_final.csv"
data = readData(filename)

subset = data[['social_isolation_score', 
                  'face_to_face_social_hours_weekly' ,
                  'sleep_hours', 
                  'exercise_hours_weekly', 
                  'daily_gaming_hours',
                  'academic_work_performance',
                  'gaming_addiction_risk_level']]

#subset = subset.to_numpy()
spearman = subset.corr(method='spearman')
spearman.to_csv("../data/subset_spearman_table.csv")